# Nootebook for correction 

p60 top lists


Goal: Fill the gaps on the following text:


To identify a robust muscle ageing signature, we focused on the feature importance rankings of the best-performing model (CatBoost 90:10). We established the **top 50** genes by mean absolute SHAP value as the threshold for global significance. 

Analysis of the validation set revealed that global importance was highly representative of individual ageing profiles; of the global **top 50** genes, \_\_ were consistently ranked within the individual **top 50** lists for at least \_\_\% of the cohort. This overlap suggests a consistent molecular backbone of muscle ageing across diverse individuals. 

In contrast, feature importance in the Ridge 90:10 model was significantly more distributed. While the model identified 384 features with non-zero coefficients, the weight was not concentrated in a small core signature. When applying the same **top 50** threshold used for CatBoost, only \_\_ genes from the Ridge global **top 50** list were consistently identified in the individual rankings across the cohort. This comparison highlights the advantage of the non-linear CatBoost model in capturing a concentrated set of high-impact biological signals compared to the more diffuse importance of the linear baseline.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import shap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score
import pickle

Let's load all the shap values of the predition models at globa level.
Each column is a gene, each row is an individual.  108 individuals totoal.

In [2]:
shap_catboost_df_40 = pd.read_csv("shap_values_by_gene_Catboost_40_extended.csv")
shap_ridge_df_40  = pd.read_csv("shap_values_by_gene_Ridge_40_extended.csv")
shap_catboost_df_10 = pd.read_csv("shap_values_by_gene_Catboost_10_extended.csv")
shap_ridge_df_10 = pd.read_csv("shap_values_by_gene_Ridge_10_extended.csv")

shap_values_dict = {
    'catboost_40': shap_catboost_df_40,
    'ridge_40': shap_ridge_df_40,
    'catboost_10': shap_catboost_df_10,
    'ridge_10': shap_ridge_df_10
}

In [3]:
cuttoff = 50

### 1.- Get the top 50 Shap values

In [4]:
# Function to get the top 50 genes with highest SHAP values and count their presence across individuals
def get_frequent_shap(shap_df, top_n=50):
    # Calculate mean absolute SHAP values for each gene
    mean_shap_values = shap_df.abs().mean().sort_values(ascending=False)
    
    # Get the top N genes
    top_genes = mean_shap_values.head(top_n).index.tolist()
    return top_genes


In [38]:
# Get the top 50 genes for each model
top_genes_global = {}
for model_name, shap_df in shap_values_dict.items():
    top_genes = get_frequent_shap(shap_df, top_n=cuttoff)
    top_genes_global[model_name] = top_genes



In [40]:
top_genes_global

{'catboost_40': ['STUM',
  'MTFR1',
  'HNRNPM',
  'WDR74',
  'CFD',
  'UBE2H',
  'SLC16A3',
  'CA3',
  'DAPK2',
  'EIF5A',
  'HSPB6',
  'ENSG00000262855.1',
  'ABCD4',
  'FBXO31',
  'TSPAN7',
  'ENSG00000285043.1',
  'JUN',
  'KLC1',
  'ZNF710',
  'STIM1',
  'HMGB2',
  'PTP4A3',
  'ILF3',
  'MORF4L2',
  'FEZ2',
  'ACIN1',
  'SUN1',
  'SDHAP1',
  'ENSG00000234441.1',
  'HBA1',
  'MYH8',
  'H3-3B',
  'UBFD1',
  'OR7E47P',
  'PERM1',
  'MICAL3',
  'SEPTIN2',
  'RNA5SP460',
  'EIF3B',
  'ENSG00000187186.14',
  'HYI',
  'FAM174B',
  'MTSS1',
  'NSFL1C',
  'HCFC1R1',
  'C12orf75',
  'MYO1B',
  'LONP2',
  'FAM177A1',
  'TSC22D1'],
 'ridge_40': ['MYL2',
  'MT-CO2',
  'MT-CO1',
  'MT-ATP6',
  'TTN',
  'MT-CO3',
  'MT-ND2',
  'MYBPC1',
  'TNNC1',
  'ENSG00000269028.3',
  'NEB',
  'ENO3',
  'MTATP6P1',
  'MB',
  'TNNI1',
  'MT-ND3',
  'TPM3',
  'ENSG00000225840.2',
  'RPL41',
  'MT-ATP8',
  'FHL1',
  'PDLIM3',
  'RPL37A',
  'MTND1P23',
  'MT-ND4',
  'RPS24',
  'MT-ND1',
  'YBX3',
  'PCSK5',
  'MY

### 2.- Count the frequency of the union of the 4 models in each of them.

In [41]:
# Count the frequency of the union of the 4 models in each of them. Max value is 4.
from collections import Counter
all_top_genes = []
for genes in top_genes_global.values():
    all_top_genes.extend(genes)

gene_counts = Counter(all_top_genes)
gene_frequency_df = pd.DataFrame.from_dict(gene_counts, orient='index', columns=['Frequency'])
gene_frequency_df = gene_frequency_df.sort_values(by='Frequency', ascending=False)
gene_frequency_df

,Frequency
EEF2,2
RNA5-8SP6,2
DAPK2,2
HSPB6,2
ENSG00000262855.1,2
...,...
CALU,1
BMI1,1
ANKRD1,1
TPM2,1


#### 2.1.- Count the frequency of the union of the catboost 10 and ridge 10.

In [42]:
# Get the intersection of top genes between CatBoost 10 and Ridge 10 models
top_genes_catboost_10 = set(top_genes_global['catboost_10'])
top_genes_ridge_10 = set(top_genes_global['ridge_10'])
intersection_10 = top_genes_catboost_10.intersection(top_genes_ridge_10)
len(intersection_10)

0

#### 2.2.- Count the frequency of the union of the catboost 10 and  catboost 40

In [43]:
# Get the intersection of top genes between CatBoost 10 and  CatBoost 40 models
top_genes_catboost_40 = set(top_genes_global['catboost_40'])
intersection_catboost = top_genes_catboost_10.intersection(top_genes_catboost_40)
len(intersection_catboost)

30

#### 2.3.- Count the frequency of the union of the ridge 10 and  ridge 40

In [44]:
# Get the intersection of top genes between ridge 10 and  ridge 40 models
top_genes_ridge_40 = set(top_genes_global['ridge_40'])
intersection_ridge = top_genes_ridge_10.intersection(top_genes_ridge_40)
len(intersection_ridge)


40

#### 2.4.- Count the frequency of the union of the catboost 40 and ridge 40.

In [45]:
# Get the frequency of the intersection of the catboost 40 and ridge 40.
intersection_40 = top_genes_catboost_40.intersection(top_genes_ridge_40)
len(intersection_40)


0

### 3.- Count individuals top 50 genes
Here I get for each individual, what were their top 50 leading genes

In [46]:
# For each individual, get their top 50 leading genes in each model. Id is the index of the individual.
def get_individual_top_genes(shap_df, top_n=50):
    individual_top_genes = {}
    for idx in shap_df.index:
        individual_shap_values = shap_df.loc[idx]
        top_genes = individual_shap_values.abs().sort_values(ascending=False).head(top_n).index.tolist()
        individual_top_genes[idx] = top_genes
    return individual_top_genes

In [47]:
individual_top_genes_dict = {}
for model_name, shap_df in shap_values_dict.items():
    individual_top_genes = get_individual_top_genes(shap_df, top_n=cuttoff)
    individual_top_genes_dict[model_name] = individual_top_genes



In [48]:
len(individual_top_genes_dict['catboost_10'][0])

50

#### 3.1.- Get intersection of the top 50 genes

In [49]:
# For each model, get the intersection of top 50 genes across all individuals
individuals_top_genes_intersection = {}
for model_name, individual_genes in individual_top_genes_dict.items():
    all_individuals_genes = [set(genes) for genes in individual_genes.values()]
    intersection_genes = set.intersection(*all_individuals_genes)
    individuals_top_genes_intersection[model_name] = intersection_genes


In [50]:
# Get the number of genes in the intersection for each model
intersection_individual_num = {model: len(genes) for model, genes in individuals_top_genes_intersection.items()}
individuals_top_genes_intersection['catboost_10']

{'ABCC5',
 'ACSF2',
 'BLCAP',
 'C12orf75',
 'ENSG00000187186.14',
 'ENSG00000234441.1',
 'ENSG00000262855.1',
 'ENSG00000285043.1',
 'H3-3B',
 'HSPB6',
 'IFT172',
 'MICU1',
 'MTFR1',
 'MYH8',
 'NSFL1C',
 'NT5C2',
 'SLC16A3',
 'STUM',
 'ZNF710'}

### 4.- Get how many of the top 50 genes Global intersect with the  invidduals

In [51]:
# Get the individuals_top_genes_intersection intersection with the global top genes top_genes_global
catboost_10_top_genes_set = set(top_genes_global['catboost_10'])
print(len(catboost_10_top_genes_set))
catboost_10_individuals_intersection = individuals_top_genes_intersection['catboost_10']


50


In [52]:
ridge_10_top_genes_set = set(top_genes_global['ridge_10'])
ridge_10_individuals_intersection = individuals_top_genes_intersection['ridge_10']

In [53]:
frequent_genes_catboost_10 = catboost_10_top_genes_set.intersection(catboost_10_individuals_intersection)
frequent_genes_catboost_10

{'ABCC5',
 'ACSF2',
 'BLCAP',
 'C12orf75',
 'ENSG00000187186.14',
 'ENSG00000234441.1',
 'ENSG00000262855.1',
 'ENSG00000285043.1',
 'H3-3B',
 'HSPB6',
 'IFT172',
 'MICU1',
 'MTFR1',
 'MYH8',
 'NSFL1C',
 'NT5C2',
 'SLC16A3',
 'STUM',
 'ZNF710'}

In [54]:
frequent_genes_ridge_10 = ridge_10_top_genes_set.intersection(ridge_10_individuals_intersection)
frequent_genes_ridge_10

set()

In [55]:
len(frequent_genes_catboost_10)

19

In [56]:
len(frequent_genes_ridge_10)

0

### 5.- Get Unique to one individual genes

In [57]:
# get union of all genes in individual_top_genes_dict['catboost_10']
union_individual_top_catboost_10 = set()
for individual in individual_top_genes_dict['catboost_10']:
    union_individual_top_catboost_10.update(individual_top_genes_dict['catboost_10'][individual])
len(union_individual_top_catboost_10)

132

In [58]:
union_individual_top_ridge_10 = set()
for individual in individual_top_genes_dict['ridge_10']:
    union_individual_top_ridge_10.update(individual_top_genes_dict['ridge_10'][individual])
len(union_individual_top_ridge_10)

146

In [59]:
# Get the frequency of each gene  in union_individual_top_catboost_10 on the individuals
gene_frequency_individuals_catboost = {}
for gene in union_individual_top_catboost_10:
    count = sum(1 for individual_genes in individual_top_genes_dict['catboost_10'].values() if gene in individual_genes)
    gene_frequency_individuals_catboost[gene] = count


In [60]:
gene_frequency_individuals_ridge = {}
for gene in union_individual_top_ridge_10:
    count = sum(1 for individual_genes in individual_top_genes_dict['ridge_10'].values() if gene in individual_genes)
    gene_frequency_individuals_ridge[gene] = count

In [61]:
gene_frequency_individuals_catboost

{'AP1M2': 1,
 'UBFD1': 107,
 'COL15A1': 43,
 'ANKRD1': 24,
 'UBXN4': 1,
 'ENSG00000225840.2': 15,
 'ENSG00000234441.1': 108,
 'TXNRD1': 3,
 'TSC22D1': 106,
 'RSRC2': 2,
 'ILF3': 107,
 'MTFR1': 108,
 'JUN': 47,
 'ENSG00000260212.1': 6,
 'TPRXL': 1,
 'TSG101': 2,
 'BSDC1': 1,
 'MCM8': 3,
 'SRM': 1,
 'NUDT4P2': 2,
 'PDE4DIPP8': 2,
 'ENSG00000285043.1': 108,
 'CCDC14': 27,
 'RNF212B': 1,
 'MICAL3': 107,
 'PNO1': 1,
 'ZNF710': 108,
 'RBM48': 2,
 'PEX11B': 5,
 'SMTNL2': 18,
 'DHRSX': 16,
 'ACSF2': 108,
 'ZNF875': 93,
 'BRMS1L': 16,
 'C9orf78': 1,
 'RNA5SP148': 58,
 'PEX3': 1,
 'STUM': 108,
 'ANKRD44': 8,
 'RNA5SP99': 3,
 'ANO6': 1,
 'POLR2G': 5,
 'ANP32A': 1,
 'OGDH': 16,
 'HBB': 1,
 'UGGT1': 6,
 'SCLY': 1,
 'CHCHD10': 92,
 'EPN2': 4,
 'SUN1': 56,
 'MICU1': 108,
 'TFDP2': 11,
 'SMARCE1': 26,
 'PTP4A3': 95,
 'MYH8': 108,
 'FBXO31': 24,
 'RAD51AP1': 17,
 'TTC3': 7,
 'RNA5SP213': 7,
 'ANKRD28': 3,
 'RECQL': 2,
 'CALU': 63,
 'LDHA': 10,
 'UQCRC1': 2,
 'HMGB2': 101,
 'MTLN': 96,
 'OR7E47P': 66,
 

In [62]:
gene_frequency_individuals_ridge

{'PDAP1': 2,
 'SLC25A4': 4,
 'TPM3': 96,
 'MTND1P23': 42,
 'ENSG00000225840.2': 81,
 'CFL2': 30,
 'ENSG00000269028.3': 94,
 'MYL12A': 53,
 'ENSG00000260212.1': 88,
 'ENSG00000270188.1': 14,
 'RPL34': 22,
 'HNRNPH1': 1,
 'MT-CO1': 100,
 'RPL7': 57,
 'ENSG00000255823.4': 50,
 'RNA5SP473': 7,
 'RNA5SP241': 3,
 'TTN': 88,
 'PPDPFL': 3,
 'TRAJ50': 1,
 'B2M': 5,
 'AMPD1': 1,
 'RPL5': 29,
 'KIAA1328': 6,
 'MYL2': 106,
 'ACTA1': 72,
 'NME7': 11,
 'GABRP': 12,
 'TPM1': 3,
 'NMRK2': 3,
 'PCSK5': 64,
 'TNNT1': 100,
 'CACNA2D1': 1,
 'PPP1R1A': 18,
 'TNNI2': 12,
 'HSPB1': 1,
 'TBL1XR1': 2,
 'RNA5-8SP6': 53,
 'MYL3': 7,
 'FHL1': 35,
 'EIF3C': 2,
 'MT-ND3': 101,
 'RPL36A': 40,
 'TNNC1': 106,
 'ENSG00000256045.2': 13,
 'RNA5SP501': 3,
 'GNAS': 2,
 'TNNI1': 98,
 'RNA5SP389': 3,
 'HBB': 15,
 'RPS2': 2,
 'RYR3': 1,
 'RPL13A': 36,
 'PKM': 5,
 'ATP2A2': 91,
 'GAPDH': 16,
 'RPL9': 6,
 'HBA2': 38,
 'RPL37A': 84,
 'MT-ND2': 95,
 'MYH2': 17,
 'RPS24': 78,
 'EEF2': 49,
 'MYL1': 5,
 'BLOC1S6': 16,
 'RPL38': 51,


In [63]:
# get genes that are unique to one individual
unique_genes_individuals_catboost = [gene for gene, freq in gene_frequency_individuals_catboost.items() if freq == 1]
len(unique_genes_individuals_catboost)

23

In [64]:
unique_genes_individuals_ridge = [gene for gene, freq in gene_frequency_individuals_ridge.items() if freq == 1]
len(unique_genes_individuals_ridge)

15

### 6.- Get proportion of genes in the overall vs specific to individual

In [65]:
# in catboost 10 model, for each individual, get the proportion of their top 50 that are also in the global top 50
proportions_individuals_in_global_catboost = {}
for individual, genes in individual_top_genes_dict['catboost_10'].items():
    intersection_w_global = set(genes).intersection(catboost_10_top_genes_set)
    proportion = len(intersection_w_global) / len(genes)
    proportions_individuals_in_global_catboost[individual] = proportion


In [66]:
proportions_individuals_in_global_ridge = {}
for individual, genes in individual_top_genes_dict['ridge_10'].items():
    intersection_w_global = set(genes).intersection(ridge_10_top_genes_set)
    proportion = len(intersection_w_global) / len(genes)
    proportions_individuals_in_global_ridge[individual] = proportion

In [67]:
# get statistics of the proportions
mean_proportion_catboost = np.mean(list(proportions_individuals_in_global_catboost.values()))
mean_proportion_catboost

0.8781481481481482

In [68]:
mean_proportion_ridge = np.mean(list(proportions_individuals_in_global_ridge.values()))
mean_proportion_ridge

0.7444444444444445

### Report

In [69]:
len(unique_genes_individuals_catboost) # Number of genes unique to one individual with catboost 10 model
len(frequent_genes_catboost_10) # Number of genes frequent across individuals and in global top genes
len(individuals_top_genes_intersection['catboost_10']) # Number of genes that appear in all individuals' top genes for catboost 10 model
len(intersection_catboost) # Number of genes in intersection between catboost 10 and catboost 40 models
len(intersection_ridge) # Number of genes in intersection between ridge 10 and ridge 40 models
len(intersection_10) # Number of genes in intersection between catboost 10 and ridge 10 models
len(intersection_40) # Number of genes in intersection between catboost 40 and ridge 40 models
mean_proportion_catboost # Mean proportion of individual top genes in global top genes


0.8781481481481482

In [70]:
# Report
report_text_catboost = f"""  Of the top {cuttoff} genes by SHAP values: we see that {len(intersection_catboost)} were common between 
catboost in both settings (10 and 40), and {len(intersection_ridge)} were common between ridge in both settings (10 and 40).
When focusing on the feature importance rankings of the best-performing model (CatBoost 90:10), 
we find that {len(individuals_top_genes_intersection['catboost_10'])} genes are present in all individuals as top {cuttoff} genes, 
while {len(frequent_genes_catboost_10)} are also part of the overall top {cuttoff} genes.
Additionally,  we can see that {len(unique_genes_individuals_catboost)} genes are unique to a single individual, 
in average, of the top {cuttoff} of each individual, {100*mean_proportion_catboost:.2f}% (SD: {np.std(list(proportions_individuals_in_global_catboost.values()), ddof=1):.2f}) 
of the genes are also of the overall top {cuttoff} genes."""

In [71]:
report_text_catboost

'  Of the top 50 genes by SHAP values: we see that 30 were common between \ncatboost in both settings (10 and 40), and 40 were common between ridge in both settings (10 and 40).\nWhen focusing on the feature importance rankings of the best-performing model (CatBoost 90:10), \nwe find that 19 genes are present in all individuals as top 50 genes, \nwhile 19 are also part of the overall top 50 genes.\nAdditionally,  we can see that 23 genes are unique to a single individual, \nin average, of the top 50 of each individual, 87.81% (SD: 0.03) \nof the genes are also of the overall top 50 genes.'

In [72]:
report_text_ridge = f"""  Of the top {cuttoff} genes by SHAP values: we see that {len(intersection_catboost)} were common between 
catboost in both settings (10 and 40), and {len(intersection_ridge)} were common between ridge in both settings (10 and 40).
When focusing on the feature importance rankings of the ridge model, 
we find that {len(individuals_top_genes_intersection['ridge_10'])} genes are present in all individuals as top {cuttoff} genes, 
while {len(frequent_genes_ridge_10)} are also part of the overall top {cuttoff} genes.
Additionally,  we can see that {len(unique_genes_individuals_ridge)} genes are unique to a single individual, 
in average, of the top {cuttoff} of each individual, {100*mean_proportion_ridge:.2f}% (SD: {np.std(list(proportions_individuals_in_global_ridge.values()), ddof=1):.2f}) 
of the genes are also of the overall top {cuttoff} genes."""
report_text_ridge

'  Of the top 50 genes by SHAP values: we see that 30 were common between \ncatboost in both settings (10 and 40), and 40 were common between ridge in both settings (10 and 40).\nWhen focusing on the feature importance rankings of the ridge model, \nwe find that 0 genes are present in all individuals as top 50 genes, \nwhile 0 are also part of the overall top 50 genes.\nAdditionally,  we can see that 15 genes are unique to a single individual, \nin average, of the top 50 of each individual, 74.44% (SD: 0.07) \nof the genes are also of the overall top 50 genes.'